# 05 — Cross-modal classification & information-preservation analysis

This notebook supplies the **quantitative computational-biophysics evidence**
that the sonification mapping in notebook 02 produces a state-discriminative,
information-preserving auditory representation of the beta2AR MD trajectories
analysed in notebook 01.

It directly tests the two **pairwise comparisons** the study was designed
around:

| Pair | State A | State B | Biophysical question |
|---|---|---|---|
| **activation** | inactive | active (apo) | Does receptor activation produce an auditory signature? |
| **ligand**     | active (apo) | active + orthosteric agonist | Does ligand binding further perturb the auditory signature within the active ensemble? |

Concretely it builds the result tables and figures that the manuscript will
cite to support the claim:

> *Under an identical sonification mapping, the mapping of activation-related
> geometric features onto musical parameters preserves the MD-state
> information in a quantifiable, timbre-independent manner for both the
> activation contrast (inactive vs active) and the ligand-binding contrast
> (active apo vs active + ligand).*

### Analyses

1. **MD-side statistical tests** — pair × feature Mann–Whitney U, Cliff's
   delta, Benjamini–Hochberg-corrected p-values. **Per pair.** Confirms
   the MD features themselves differ between the two states in each pair.
2. **Acoustic feature extraction** — every WAV produced in notebook 02 is
   windowed (1-s windows, 0.5-s hop) and described by MFCC(13), spectral
   centroid / rolloff / bandwidth, RMS energy, zero-crossing rate. This
   expands the per-state N from 1 to ~hundreds.
3. **Acoustic-side statistical tests** — same MWU framework on each audio
   descriptor. **Per pair.**
4. **PCA of audio features** — three-state scatter; visual orientation.
5. **Per-pair binary classification** *(primary result)* — for each pair a
   separate Random Forest is trained to discriminate the two states from
   audio features alone (chance = 0.500). Reported per-instrument and
   pooled across instruments.
6. **Per-pair leave-one-instrument-out (LOIO) generalisation** — train on
   two instruments, test on the held-out third, again **per pair**. Tests
   timbre-independence of each pair's MD-state signal.
7. **Per-pair MD-feature baseline classifier** — same RF but trained on
   the raw MD features for each pair; gives the upper-bound accuracy the
   audio classifier could possibly approach. The audio-to-MD accuracy
   ratio is reported as an **information-retention metric** per pair.
8. **Per-pair Canonical Correlation Analysis** — multivariate correlation
   between MD and audio feature spaces restricted to each pair; the first
   canonical correlation summarises how much of the pair's MD variance is
   captured in audio.
9. **3-class supplementary analyses** — full 3-state classifier, LOIO,
   MD baseline, CCA, all kept for completeness and to show the framework
   generalises beyond the two designed pairs.

All outputs are written under `outputs/tables/` and `outputs/figures/` so
they can be picked up directly by the manuscript.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install dependencies


In [ ]:
!pip -q install librosa scikit-learn scipy pandas matplotlib seaborn


## 3. Imports, paths, publication-style matplotlib


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import librosa
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, balanced_accuracy_score)
from sklearn.cross_decomposition import CCA

PROJECT_DIR   = Path('/content/drive/MyDrive/GPCR_Sonification')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
AUDIO_DIR     = PROJECT_DIR / 'outputs' / 'audio'
FIG_DIR       = PROJECT_DIR / 'outputs' / 'figures'
TABLE_DIR     = PROJECT_DIR / 'outputs' / 'tables'
for d in [FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Publication-style matplotlib settings, identical to notebooks 01-03
# so every figure across the project shares the same look-and-feel.
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.titleweight':  'bold',
    'axes.labelsize':    10,
    'axes.linewidth':    0.8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        110,
    'savefig.dpi':       600,
    'savefig.bbox':      'tight',
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ---- Colorblind-safe palette, identical across all notebooks ----
# State colours (Okabe-Ito subset — same in notebooks 01-04)
STATE_ORDER   = ['inactive', 'active', 'active_ligand_bound']
STATE_COLORS  = {'inactive':             '#0072B2',   # blue
                 'active':               '#E69F00',   # orange
                 'active_ligand_bound':  '#009E73'}   # green
STATE_DISPLAY = {'inactive':             'inactive',
                 'active':               'active',
                 'active_ligand_bound':  'active + ligand'}

# PAIR colours (Okabe-Ito categorical, deliberately DIFFERENT from
# state colours so pair-level plots like the CCA bar chart never
# semantically collide with state-level plots like the PCA scatter).
PAIR_COLORS = {'activation': '#56B4E9',   # sky blue
               'ligand':     '#CC79A7',   # reddish purple
               '3_state':    '#999999'}   # neutral grey
PAIR_DISPLAY = {'activation': 'activation pair',
                'ligand':     'ligand pair',
                '3_state':    'all three states'}

# Heatmap colormap shared across confusion matrices in this notebook.
# 'mako_r' is seaborn's perceptually-uniform sequential going light->dark,
# pairs visually with the Okabe-Ito categorical palette without clashing.
HEATMAP_CMAP = 'mako_r'

INSTRUMENTS = ['piano', 'violin', 'flute']
INSTRUMENT_MARKERS = {'piano': 'o', 'violin': 's', 'flute': '^'}
RNG = np.random.default_rng(42)

def savefig_multi(fig, base, formats=('png', 'pdf', 'svg')):
    for fmt in formats:
        fig.savefig(f'{base}.{fmt}')

## 4. Load MD features and mapping tables


In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'all_features.csv')
df['state'] = pd.Categorical(df['state'], categories=STATE_ORDER, ordered=True)
print('MD features loaded:', df.shape, 'states:', df['state'].unique().tolist())

mapping_tables = {}
for s in STATE_ORDER:
    f = PROCESSED_DIR / f'{s}_sonification_mapping.csv'
    if f.exists():
        mapping_tables[s] = pd.read_csv(f)
print('Mapping tables    :', list(mapping_tables))

PAIRS = {
    'activation': {'states': ['inactive', 'active']},
    'ligand':     {'states': ['active', 'active_ligand_bound']},
}


## 5. MD-side statistical tests

For each pair x each MD feature, run a two-sided Mann-Whitney U test and
compute Cliff's delta as a non-parametric effect size. All p-values are
combined per analysis with Benjamini-Hochberg FDR correction.


In [ ]:
def cliffs_delta(x, y):
    """Cliff's delta (non-parametric, robust effect size in [-1, +1])."""
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (nx * ny)

def bh_fdr(pvals):
    """Benjamini-Hochberg FDR adjustment."""
    p = np.asarray(pvals, float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(adj)
    out[order] = np.clip(adj, 0, 1)
    return out

MD_FEATURES_OF_INTEREST = [
    'RMSD_Ca_A', 'TM3_TM6_distance_A', 'NPxxY_RMSD_A', 'DRY_ionic_lock_A',
    'Ligand_min_distance_A', 'Ligand_contact_count',
]

rows = []
for pair_name, pair_cfg in PAIRS.items():
    sa, sb = pair_cfg['states']
    for feat in MD_FEATURES_OF_INTEREST:
        if feat not in df.columns:
            continue
        x = df.loc[df['state'] == sa, feat].dropna().values
        y = df.loc[df['state'] == sb, feat].dropna().values
        if len(x) < 5 or len(y) < 5:
            rows.append({'pair': pair_name, 'feature': feat,
                         'n_a': len(x), 'n_b': len(y),
                         'U': np.nan, 'p': np.nan, 'cliffs_delta': np.nan})
            continue
        u, p = stats.mannwhitneyu(x, y, alternative='two-sided')
        rows.append({
            'pair':         pair_name, 'feature': feat,
            'state_a':      sa, 'state_b': sb,
            'n_a':          len(x), 'n_b': len(y),
            'mean_a':       float(np.mean(x)),
            'mean_b':       float(np.mean(y)),
            'std_a':        float(np.std(x, ddof=1)),
            'std_b':        float(np.std(y, ddof=1)),
            'U':            float(u),
            'p':            float(p),
            'cliffs_delta': float(cliffs_delta(x, y)),
        })

md_stats = pd.DataFrame(rows)
md_stats['p_bh'] = bh_fdr(md_stats['p'].fillna(1.0).values)
md_stats['significant_q0.05'] = md_stats['p_bh'] < 0.05
md_stats.to_csv(TABLE_DIR / 'md_statistical_tests.csv', index=False)
print('Saved md_statistical_tests.csv')
md_stats[['pair','feature','mean_a','mean_b','U','p','p_bh',
          'cliffs_delta','significant_q0.05']]


## 6. Acoustic feature extraction

Every WAV from notebook 02 (`{state}_{instrument}.wav`, 9 files in total)
is sliced into 1-s analysis windows with 0.5-s hop. From each window we
extract a 28-dimensional descriptor vector:

- 13 MFCC means + 13 MFCC stds
- spectral centroid (mean, std)
- spectral rolloff (mean, std)
- spectral bandwidth (mean, std)
- RMS energy (mean, std)
- zero-crossing rate (mean, std)

This expands the per-state sample size from 1 WAV to hundreds of windowed
observations, which is what makes the downstream classifier statistics
meaningful.


In [ ]:
WINDOW_S = 1.0   # window length (seconds)
HOP_S    = 0.5   # hop between consecutive windows

def descriptor_for_segment(y_seg, sr):
    if len(y_seg) < int(0.1 * sr):
        return None
    mfcc = librosa.feature.mfcc(y=y_seg, sr=sr, n_mfcc=13)
    sc   = librosa.feature.spectral_centroid(y=y_seg, sr=sr)[0]
    sr_  = librosa.feature.spectral_rolloff(y=y_seg, sr=sr, roll_percent=0.85)[0]
    sb   = librosa.feature.spectral_bandwidth(y=y_seg, sr=sr)[0]
    rms  = librosa.feature.rms(y=y_seg)[0]
    zcr  = librosa.feature.zero_crossing_rate(y=y_seg)[0]
    feat = {}
    for i in range(13):
        feat[f'mfcc{i+1}_mean'] = float(np.mean(mfcc[i]))
        feat[f'mfcc{i+1}_std']  = float(np.std(mfcc[i]))
    feat['spec_centroid_mean']  = float(np.mean(sc))
    feat['spec_centroid_std']   = float(np.std(sc))
    feat['spec_rolloff_mean']   = float(np.mean(sr_))
    feat['spec_rolloff_std']    = float(np.std(sr_))
    feat['spec_bandwidth_mean'] = float(np.mean(sb))
    feat['spec_bandwidth_std']  = float(np.std(sb))
    feat['rms_mean']            = float(np.mean(rms))
    feat['rms_std']             = float(np.std(rms))
    feat['zcr_mean']            = float(np.mean(zcr))
    feat['zcr_std']             = float(np.std(zcr))
    return feat

rows = []
for state in STATE_ORDER:
    for instr in INSTRUMENTS:
        wav = AUDIO_DIR / f'{state}_{instr}.wav'
        if not wav.exists():
            print('[SKIP]', wav.name)
            continue
        y, sr = librosa.load(str(wav), sr=None, mono=True)
        win = int(WINDOW_S * sr)
        hop = int(HOP_S * sr)
        n_windows = max(0, 1 + (len(y) - win) // hop)
        print(f'  {wav.name:42s}  sr={sr}  dur={len(y)/sr:6.1f}s  windows={n_windows}')
        for w in range(n_windows):
            seg = y[w*hop : w*hop + win]
            desc = descriptor_for_segment(seg, sr)
            if desc is None:
                continue
            desc.update({
                'state':           state,
                'instrument':      instr,
                'window_idx':      w,
                'window_start_s':  w * HOP_S,
            })
            rows.append(desc)

audio_feats = pd.DataFrame(rows)
audio_feats['state'] = pd.Categorical(audio_feats['state'],
                                      categories=STATE_ORDER, ordered=True)
print('\nAudio feature matrix:', audio_feats.shape)
print(audio_feats.groupby(['state','instrument']).size().unstack())
audio_feats.to_csv(TABLE_DIR / 'audio_features_windowed.csv', index=False)
print('Saved audio_features_windowed.csv')
audio_feats.head()


## 7. Acoustic-side statistical tests

Repeat the MWU + Cliff's delta pipeline on each scalar audio descriptor.
Windows are pooled across instruments per state (instrument is a nuisance
variable here; pooled MWU tests whether the state difference is
detectable in audio even after collapsing timbre).


In [ ]:
audio_descriptors = [c for c in audio_feats.columns
                     if c not in ('state', 'instrument',
                                  'window_idx', 'window_start_s')]
print(f'Acoustic descriptors: {len(audio_descriptors)}')

rows = []
for pair_name, pair_cfg in PAIRS.items():
    sa, sb = pair_cfg['states']
    for feat in audio_descriptors:
        x = audio_feats.loc[audio_feats['state'] == sa, feat].dropna().values
        y = audio_feats.loc[audio_feats['state'] == sb, feat].dropna().values
        if len(x) < 10 or len(y) < 10:
            continue
        u, p = stats.mannwhitneyu(x, y, alternative='two-sided')
        rows.append({
            'pair':         pair_name, 'feature': feat,
            'state_a':      sa, 'state_b': sb,
            'n_a':          len(x), 'n_b': len(y),
            'mean_a':       float(np.mean(x)),
            'mean_b':       float(np.mean(y)),
            'U':            float(u),
            'p':            float(p),
            'cliffs_delta': float(cliffs_delta(x, y)),
        })

audio_stats = pd.DataFrame(rows)
audio_stats['p_bh'] = bh_fdr(audio_stats['p'].fillna(1.0).values)
audio_stats['significant_q0.05'] = audio_stats['p_bh'] < 0.05
audio_stats.to_csv(TABLE_DIR / 'audio_statistical_tests.csv', index=False)
print('Significant audio descriptors per pair (q<0.05):')
print(audio_stats.groupby('pair')['significant_q0.05'].sum())
audio_stats.sort_values('p').head(15)


## 8. PCA of audio features

Project the 28-D windowed audio feature matrix onto 2 PCs. If the three
states form visibly separated clouds, the auditory representation has
state-discriminative structure.


In [ ]:
X   = audio_feats[audio_descriptors].values
y   = audio_feats['state'].astype(str).values
ins = audio_feats['instrument'].values

Xs  = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
Z   = pca.fit_transform(Xs)

fig, ax = plt.subplots(figsize=(5.8, 4.8), constrained_layout=True)
for s in STATE_ORDER:
    for instr in INSTRUMENTS:
        mask = (y == s) & (ins == instr)
        if not mask.any():
            continue
        ax.scatter(Z[mask, 0], Z[mask, 1],
                   color=STATE_COLORS[s],
                   marker=INSTRUMENT_MARKERS[instr],
                   s=18, alpha=0.45, edgecolor='none')

# Two clean legends: state (color) on the top-left, instrument
# (marker shape) on the bottom-left.
state_handles = [plt.Line2D([0], [0], marker='o', linestyle='',
                            color=STATE_COLORS[s], label=STATE_DISPLAY[s],
                            markersize=8) for s in STATE_ORDER]
instr_handles = [plt.Line2D([0], [0], marker=INSTRUMENT_MARKERS[i],
                            linestyle='', color='gray', label=i,
                            markersize=8) for i in INSTRUMENTS]
leg1 = ax.legend(handles=state_handles, title='state',
                 loc='upper left', fontsize=8, frameon=True,
                 framealpha=0.9, edgecolor='none')
ax.add_artist(leg1)
ax.legend(handles=instr_handles, title='instrument',
          loc='lower left', fontsize=8, frameon=True,
          framealpha=0.9, edgecolor='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of windowed audio feature space')
savefig_multi(fig, FIG_DIR / 'fig_audio_features_pca')
plt.show()
print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.3f}, '
      f'cum={pca.explained_variance_ratio_[:2].sum():.3f}')


## 9. Cross-modal classification — Random Forest on audio features

**Primary result, per pair (binary, chance = 0.500):** for each of the
two designed pairs (`activation`, `ligand`) we train a separate Random
Forest classifier that sees only audio features and must discriminate
the two states. Reported per-instrument and pooled across the three
instruments.

**Supplementary (3-class, chance = 0.333):** a single RF discriminating
all three states simultaneously, kept for completeness.

This is the *primary computational evidence* that the audio carries the
MD state information.


In [ ]:
def evaluate_classifier(X, y, label, classes):
    """Stratified 5-fold CV with RandomForest. Works for binary AND
    multi-class — pass the class label list explicitly so the confusion
    matrix is always reported in the desired order."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X, y, cv=skf, scoring='balanced_accuracy')
    preds  = cross_val_predict(clf, X, y, cv=skf)
    cm     = confusion_matrix(y, preds, labels=classes)
    rep    = classification_report(y, preds, labels=classes,
                                   output_dict=True, zero_division=0)
    print(f'  {label:50s} balanced acc = {scores.mean():.3f} +/- {scores.std():.3f}')
    return {
        'label':              label,
        'n_samples':          len(X),
        'mean_balanced_acc':  float(scores.mean()),
        'std_balanced_acc':   float(scores.std()),
        'fold_scores':        scores.tolist(),
        'confusion_matrix':   cm.tolist(),
        'per_class_f1':       {c: rep[c]['f1-score']
                               for c in classes if c in rep},
    }, cm

# ============================================================
# PRIMARY: per-pair binary classification (chance = 0.500)
# ============================================================
print('=== Per-pair binary classification (PRIMARY; chance = 0.500) ===')
binary_results = []
for pair_name, pair_cfg in PAIRS.items():
    sa, sb = pair_cfg['states']
    pair_audio = audio_feats[audio_feats['state'].isin([sa, sb])]

    # Per-instrument
    for instr in INSTRUMENTS:
        sel = pair_audio['instrument'] == instr
        if not sel.any():
            continue
        Xp = StandardScaler().fit_transform(
            pair_audio.loc[sel, audio_descriptors].values)
        yp = pair_audio.loc[sel, 'state'].astype(str).values
        res, _ = evaluate_classifier(Xp, yp,
                                      f'{pair_name} / {instr}', [sa, sb])
        res['pair'] = pair_name
        res['instrument'] = instr
        binary_results.append(res)

    # Pooled across instruments for this pair
    Xp = StandardScaler().fit_transform(pair_audio[audio_descriptors].values)
    yp = pair_audio['state'].astype(str).values
    res, cm_pair = evaluate_classifier(Xp, yp,
                                        f'{pair_name} / pooled', [sa, sb])
    res['pair'] = pair_name
    res['instrument'] = 'pooled'
    binary_results.append(res)

pd.DataFrame(binary_results).to_csv(
    TABLE_DIR / 'binary_classification_results.csv', index=False)
print('Saved binary_classification_results.csv')

# Per-pair pooled confusion matrices side-by-side (Fig — main).
# Each panel is tinted with its PAIR colour for at-a-glance grouping.
fig, axes = plt.subplots(1, len(PAIRS), figsize=(5.2 * len(PAIRS), 4.6),
                         constrained_layout=True)
if len(PAIRS) == 1:
    axes = [axes]
for ax, (pair_name, pair_cfg) in zip(axes, PAIRS.items()):
    sa, sb = pair_cfg['states']
    res = next(r for r in binary_results
               if r['pair'] == pair_name and r['instrument'] == 'pooled')
    cm = np.array(res['confusion_matrix'])
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap=HEATMAP_CMAP,
                vmin=0.0, vmax=1.0,
                xticklabels=[STATE_DISPLAY[sa], STATE_DISPLAY[sb]],
                yticklabels=[STATE_DISPLAY[sa], STATE_DISPLAY[sb]],
                cbar_kws={'label': 'row-normalised proportion'}, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True' if ax is axes[0] else '')
    ax.set_title(f'{PAIR_DISPLAY[pair_name]} (pooled)\n'
                 f'balanced acc = {res["mean_balanced_acc"]:.3f} '
                 f'± {res["std_balanced_acc"]:.3f}',
                 color=PAIR_COLORS[pair_name])
fig.suptitle('Per-pair binary classification — pooled audio features',
             fontweight='bold')
savefig_multi(fig, FIG_DIR / 'fig_classification_confusion_per_pair')
plt.show()

# ============================================================
# SUPPLEMENTARY: 3-class classification (chance = 0.333)
# ============================================================
print('\n=== Supplementary: 3-class classification (chance = 0.333) ===')
three_class_results = []
for instr in INSTRUMENTS:
    sel = audio_feats['instrument'] == instr
    Xi = StandardScaler().fit_transform(
        audio_feats.loc[sel, audio_descriptors].values)
    yi = audio_feats.loc[sel, 'state'].astype(str).values
    res, cm = evaluate_classifier(Xi, yi, f'3-class / {instr}', STATE_ORDER)
    three_class_results.append(res)

Xall = StandardScaler().fit_transform(audio_feats[audio_descriptors].values)
yall = audio_feats['state'].astype(str).values
res_audio, cm_all = evaluate_classifier(Xall, yall,
                                         '3-class / pooled', STATE_ORDER)
three_class_results.append(res_audio)

pd.DataFrame(three_class_results).to_csv(
    TABLE_DIR / 'three_class_classification_results.csv', index=False)

fig, ax = plt.subplots(figsize=(5.4, 4.6), constrained_layout=True)
cm_norm = cm_all.astype(float) / cm_all.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=cm_all, fmt='d', cmap=HEATMAP_CMAP,
            vmin=0.0, vmax=1.0,
            xticklabels=[STATE_DISPLAY[s] for s in STATE_ORDER],
            yticklabels=[STATE_DISPLAY[s] for s in STATE_ORDER],
            cbar_kws={'label': 'row-normalised proportion'}, ax=ax)
ax.set_xlabel('Predicted state')
ax.set_ylabel('True state')
ax.set_title('Pooled audio RF — 3-class confusion (supplementary)')
savefig_multi(fig, FIG_DIR / 'fig_classification_confusion_3class_pooled')
plt.show()


## 10. Leave-one-instrument-out (LOIO) generalisation — per pair

For each pair we train a binary classifier on **two of the three
instruments** (e.g. piano + violin) and test on the held-out third
(flute). If the model still generalises, the MD-state signal is
**timbre-independent** — it lives in the mapped audio content, not in
the synth.

The 2 pairs × 3 held-out instruments = 6-cell grid below is the
manuscript's main timbre-robustness evidence.


In [ ]:
print('=== Per-pair LOIO (binary; chance = 0.500) ===')
loio_rows = []
for pair_name, pair_cfg in PAIRS.items():
    sa, sb = pair_cfg['states']
    pair_audio = audio_feats[audio_feats['state'].isin([sa, sb])].copy()
    for held in INSTRUMENTS:
        train_mask = pair_audio['instrument'] != held
        test_mask  = pair_audio['instrument'] == held
        if not train_mask.any() or not test_mask.any():
            continue
        X_tr = pair_audio.loc[train_mask, audio_descriptors].values
        y_tr = pair_audio.loc[train_mask, 'state'].astype(str).values
        X_te = pair_audio.loc[test_mask,  audio_descriptors].values
        y_te = pair_audio.loc[test_mask,  'state'].astype(str).values
        scaler = StandardScaler().fit(X_tr)
        X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
        clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        bal   = balanced_accuracy_score(y_te, preds)
        cm    = confusion_matrix(y_te, preds, labels=[sa, sb])
        print(f'  {pair_name:11s} held-out = {held:6s}  '
              f'balanced acc = {bal:.3f}  (n_test = {len(y_te)})')
        loio_rows.append({
            'pair':                pair_name,
            'held_out_instrument': held,
            'n_train':             int(train_mask.sum()),
            'n_test':              int(test_mask.sum()),
            'balanced_accuracy':   float(bal),
            'confusion_matrix':    cm.tolist(),
        })

loio_df = pd.DataFrame(loio_rows)
loio_df.to_csv(TABLE_DIR / 'loio_binary_classification_results.csv', index=False)
print('\nMean LOIO balanced accuracy per pair:')
print(loio_df.groupby('pair')['balanced_accuracy'].agg(['mean', 'std']))

# Visualise: pair-rows x instrument-cols grid of confusion matrices.
# Row labels are tinted with the pair's colour for at-a-glance grouping.
fig, axes = plt.subplots(len(PAIRS), len(INSTRUMENTS),
                         figsize=(4.0 * len(INSTRUMENTS), 3.6 * len(PAIRS)),
                         constrained_layout=True, squeeze=False)
for i, (pair_name, pair_cfg) in enumerate(PAIRS.items()):
    sa, sb = pair_cfg['states']
    for j, instr in enumerate(INSTRUMENTS):
        ax = axes[i, j]
        row = loio_df[(loio_df['pair'] == pair_name) &
                       (loio_df['held_out_instrument'] == instr)]
        if row.empty:
            ax.axis('off')
            continue
        cm = np.array(row.iloc[0]['confusion_matrix'])
        cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        bal = row.iloc[0]['balanced_accuracy']
        sns.heatmap(cm_norm, annot=cm, fmt='d', cmap=HEATMAP_CMAP,
                    vmin=0.0, vmax=1.0,
                    xticklabels=[STATE_DISPLAY[sa], STATE_DISPLAY[sb]],
                    yticklabels=[STATE_DISPLAY[sa], STATE_DISPLAY[sb]],
                    cbar=False, ax=ax)
        if i == 0:
            ax.set_title(f'held-out = {instr}\nbal acc = {bal:.2f}',
                         fontweight='bold')
        else:
            ax.set_title(f'bal acc = {bal:.2f}')
        if j == 0:
            ax.set_ylabel(f'{PAIR_DISPLAY[pair_name]}\n(True)',
                          fontweight='bold',
                          color=PAIR_COLORS[pair_name])
        else:
            ax.set_ylabel('')
        if i == len(PAIRS) - 1:
            ax.set_xlabel('Predicted')
        else:
            ax.set_xlabel('')

fig.suptitle('Leave-one-instrument-out generalisation — per pair',
             fontweight='bold')
savefig_multi(fig, FIG_DIR / 'fig_classification_loio_per_pair')
plt.show()

# Also keep the original 3-class LOIO as supplementary
print('\n=== Supplementary: 3-class LOIO ===')
loio_3c_rows = []
for held in INSTRUMENTS:
    train_mask = audio_feats['instrument'] != held
    test_mask  = audio_feats['instrument'] == held
    X_tr = audio_feats.loc[train_mask, audio_descriptors].values
    y_tr = audio_feats.loc[train_mask, 'state'].astype(str).values
    X_te = audio_feats.loc[test_mask,  audio_descriptors].values
    y_te = audio_feats.loc[test_mask,  'state'].astype(str).values
    scaler = StandardScaler().fit(X_tr)
    X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
    clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    bal = balanced_accuracy_score(y_te, preds)
    cm = confusion_matrix(y_te, preds, labels=STATE_ORDER)
    print(f'  3-class held-out = {held:6s}  balanced acc = {bal:.3f}')
    loio_3c_rows.append({
        'held_out_instrument': held,
        'n_train':             int(train_mask.sum()),
        'n_test':              int(test_mask.sum()),
        'balanced_accuracy':   float(bal),
        'confusion_matrix':    cm.tolist(),
    })
pd.DataFrame(loio_3c_rows).to_csv(
    TABLE_DIR / 'loio_3class_classification_results.csv', index=False)
loio_df


## 11. Per-pair MD-feature baseline + information-retention metric

For each pair we also train the same Random Forest **directly on the MD
features** (no audio in between). This gives the upper bound that the
audio classifier could possibly reach. The ratio

```
retention_ratio(pair) = audio_pooled_acc(pair) / md_baseline_acc(pair)
```

is the manuscript's single-number information-preservation metric per
pair. A value near 1.0 means the sonification preserved essentially all
of the discriminative information in the MD features.


In [ ]:
md_features_for_clf = [c for c in MD_FEATURES_OF_INTEREST if c in df.columns]

# Biological sentinel values for features that are *structurally* NaN in
# apo systems (no ligand present). Using these instead of a borrowed
# median is what stops apo data from being made to look ligand-bound
# (which would artificially LOWER the ligand-pair MD baseline).
LIGAND_FEATURE_SENTINELS = {
    'Ligand_contact_count':  0.0,   # apo: literally zero contacts
    'Ligand_min_distance_A': 30.0,  # apo: "very far" sentinel (>> typical bound ~1.8 Å)
}

def prepare_md_matrix(states, features):
    """Return (X, y) for the given list of states using only features
    that carry information in at least one of those states. Imputation:
      - features with biological sentinels (see above): NaN -> sentinel
      - other features: per-state median
      - features that are all-NaN across every state in the pair: dropped
    """
    valid = [f for f in features
             if any(df.loc[df['state'] == s, f].notna().sum() > 0
                    for s in states)]
    parts, ys = [], []
    for s in states:
        sub = df.loc[df['state'] == s, valid].copy()
        for c in valid:
            if sub[c].isna().all():
                # All NaN in THIS state — use biological sentinel if known,
                # else 0.0 as neutral fill.
                sub[c] = LIGAND_FEATURE_SENTINELS.get(c, 0.0)
            else:
                # Per-state median fill (preserves within-state distribution)
                sub[c] = sub[c].fillna(sub[c].median())
        parts.append(sub.values)
        ys.append(np.repeat(s, len(sub)))
    X = np.vstack(parts)
    y = np.concatenate(ys)
    return X, y, valid

# ============================================================
# Per-pair MD baseline (binary)
# ============================================================
print('=== Per-pair MD baseline (binary; chance = 0.500) ===')
md_baseline_rows = []
for pair_name, pair_cfg in PAIRS.items():
    sa, sb = pair_cfg['states']
    X_raw, y, used = prepare_md_matrix([sa, sb], md_features_for_clf)
    X = StandardScaler().fit_transform(X_raw)
    print(f'  [{pair_name}] using {len(used)} MD features: {used}')
    res, _ = evaluate_classifier(X, y, f'MD baseline / {pair_name}', [sa, sb])
    res['pair'] = pair_name
    res['features_used'] = used
    md_baseline_rows.append(res)

# 3-class MD baseline (supplementary)
print('\n=== Supplementary: 3-class MD baseline ===')
X_raw, y, used = prepare_md_matrix(STATE_ORDER, md_features_for_clf)
X_md_all = StandardScaler().fit_transform(X_raw)
print(f'  [3-class] using {len(used)} MD features: {used}')
res_3c, _ = evaluate_classifier(X_md_all, y, 'MD baseline / 3-class', STATE_ORDER)
res_3c['pair'] = '3_class'
res_3c['features_used'] = used
md_baseline_rows.append(res_3c)

baseline_df = pd.DataFrame(md_baseline_rows)
baseline_df.to_csv(TABLE_DIR / 'md_baseline_classification_results.csv', index=False)
print('\nSaved md_baseline_classification_results.csv')

# ============================================================
# Information-retention ratio (audio_pooled / MD_baseline) per pair
# ============================================================
print('\n=== Information-retention ratio (audio_pooled / MD_baseline) ===')
binary_df = pd.read_csv(TABLE_DIR / 'binary_classification_results.csv')
retention_rows = []
for pair_name in PAIRS:
    audio_pooled = float(binary_df[(binary_df['pair'] == pair_name) &
                                    (binary_df['instrument'] == 'pooled')]
                         ['mean_balanced_acc'].values[0])
    md_pair = float(baseline_df[baseline_df['pair'] == pair_name]
                    ['mean_balanced_acc'].values[0])
    ratio = audio_pooled / md_pair if md_pair > 0 else np.nan
    print(f'  {pair_name:11s}  audio_pooled = {audio_pooled:.3f}   '
          f'MD_baseline = {md_pair:.3f}   ratio = {ratio:.3f}')
    retention_rows.append({
        'pair':              pair_name,
        'audio_pooled_acc':  audio_pooled,
        'md_baseline_acc':   md_pair,
        'retention_ratio':   ratio,
    })

retention_df = pd.DataFrame(retention_rows)
retention_df.to_csv(TABLE_DIR / 'information_retention.csv', index=False)
print('\nSaved information_retention.csv')
retention_df


## 12. Per-pair Canonical Correlation Analysis (MD ↔ audio)

For each pair we run CCA between paired MD-feature vectors (6-D) and the
audio descriptor vectors (28-D) of the audio windows that fall on the
matching sonification step. The first canonical correlation **r₁** is
the closest single-number summary of "how much of this pair's MD
variance is encoded in audio".

A 3-state CCA is also reported as a supplementary global check.


In [ ]:
# Align MD frames to audio windows using the per-state mapping table.
# Audio start time of sonification step i:
#     start[i] = sum(duration[:i]) + i * NOTE_GAP_S
NOTE_GAP_S = 0.02

def note_start_times(mapping_df):
    d = mapping_df['duration_s'].values
    return np.concatenate([[0.0], np.cumsum(d + NOTE_GAP_S)[:-1]])

def cca_for_states(states, label):
    """Run CCA for MD <-> audio restricted to the given list of states.
    NaN-safe: drops MD features that carry no information in either state
    (all-NaN across the pair) and applies biological sentinels +
    per-state median imputation for the survivors, identical to the
    rule used by the MD-baseline classifier in the previous cell."""
    valid_features = [
        f for f in md_features_for_clf
        if any(df.loc[df['state'] == s, f].notna().sum() > 0 for s in states)
    ]
    if not valid_features:
        print(f'  [{label}] no valid MD features for this pair — skipping')
        return [], 0

    md_rows, audio_rows = [], []
    for state in states:
        if state not in mapping_tables:
            continue
        mt = mapping_tables[state]
        starts = note_start_times(mt)
        state_md = df[df['state'] == state].reset_index(drop=True)
        md_block = state_md[valid_features].copy()
        for c in valid_features:
            if md_block[c].isna().all():
                md_block[c] = LIGAND_FEATURE_SENTINELS.get(c, 0.0)
            else:
                med = md_block[c].median()
                if np.isnan(med):
                    med = LIGAND_FEATURE_SENTINELS.get(c, 0.0)
                md_block[c] = md_block[c].fillna(med)

        for instr in INSTRUMENTS:
            sub = audio_feats[(audio_feats['state'] == state) &
                              (audio_feats['instrument'] == instr)]
            for _, r in sub.iterrows():
                t_center = r['window_start_s'] + WINDOW_S / 2.0
                idx = int(np.argmin(np.abs(starts - t_center)))
                if idx >= len(md_block):
                    continue
                md_rows.append(md_block.iloc[idx].values)
                audio_rows.append(r[audio_descriptors].values)

    MD_mat = np.array(md_rows, dtype=float)
    AU_mat = np.array(audio_rows, dtype=float)

    # Safety: drop residual NaN rows and zero-variance columns
    valid_rows = ~(np.isnan(MD_mat).any(axis=1) | np.isnan(AU_mat).any(axis=1))
    if not valid_rows.all():
        n_drop = int((~valid_rows).sum())
        print(f'  [{label}] dropping {n_drop} rows with residual NaN')
        MD_mat = MD_mat[valid_rows]
        AU_mat = AU_mat[valid_rows]
    if len(MD_mat) < 30:
        print(f'  [{label}] not enough paired samples (n = {len(MD_mat)})')
        return [], 0
    md_keep = MD_mat.std(axis=0) > 1e-12
    au_keep = AU_mat.std(axis=0) > 1e-12
    if not md_keep.all():
        print(f'  [{label}] dropping {int((~md_keep).sum())} constant MD column(s)')
        MD_mat = MD_mat[:, md_keep]
    if not au_keep.all():
        print(f'  [{label}] dropping {int((~au_keep).sum())} constant audio column(s)')
        AU_mat = AU_mat[:, au_keep]

    MD_s = StandardScaler().fit_transform(MD_mat)
    AU_s = StandardScaler().fit_transform(AU_mat)

    n_comp = min(MD_s.shape[1], AU_s.shape[1], 4)
    cca = CCA(n_components=n_comp, max_iter=2000)
    cca.fit(MD_s, AU_s)
    U, V = cca.transform(MD_s, AU_s)
    r = [float(np.corrcoef(U[:, k], V[:, k])[0, 1]) for k in range(n_comp)]
    print(f'  [{label:20s}] n_paired = {len(MD_mat):4d}   '
          + '   '.join([f'r_{k+1}={rr:.3f}' for k, rr in enumerate(r)])
          + f'   features={valid_features}')
    return r, len(MD_mat)

print('=== Per-pair canonical correlations ===')
cca_rows = []
for pair_name, pair_cfg in PAIRS.items():
    r_vals, n = cca_for_states(pair_cfg['states'], pair_name)
    for k, rr in enumerate(r_vals):
        cca_rows.append({
            'pair':                  pair_name,
            'component':             k + 1,
            'canonical_correlation': rr,
            'n_paired_samples':      n,
        })

# Supplementary: 3-state CCA
print('\n=== Supplementary: 3-state CCA ===')
r_full, n_full = cca_for_states(STATE_ORDER, '3_state')
for k, rr in enumerate(r_full):
    cca_rows.append({
        'pair':                  '3_state',
        'component':             k + 1,
        'canonical_correlation': rr,
        'n_paired_samples':      n_full,
    })

cca_df = pd.DataFrame(cca_rows)
cca_df.to_csv(TABLE_DIR / 'cca_md_audio.csv', index=False)
print('\nSaved cca_md_audio.csv')

# Grouped bar plot per pair — uses the project-wide PAIR_COLORS so it
# never collides with the STATE_COLORS used elsewhere.
fig, ax = plt.subplots(figsize=(7.0, 3.8), constrained_layout=True)
pivot = cca_df.pivot(index='component', columns='pair',
                     values='canonical_correlation')
cols = [c for c in ['activation', 'ligand', '3_state'] if c in pivot.columns]
pivot = pivot[cols]
pivot.plot.bar(ax=ax, color=[PAIR_COLORS[c] for c in cols],
               edgecolor='black', linewidth=0.6, width=0.78)
ax.set_xlabel('Canonical component')
ax.set_ylabel('Canonical correlation r')
ax.set_ylim(0, 1.0)
ax.set_title('MD ↔ audio canonical correlations (per pair)')
ax.legend([PAIR_DISPLAY[c] for c in cols],
          title='', loc='best', frameon=True, framealpha=0.9,
          edgecolor='none')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    for bar in container:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.015,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=7)
savefig_multi(fig, FIG_DIR / 'fig_cca_md_audio')
plt.show()
cca_df


## 13. Summary

### Primary, per-pair outputs (manuscript main results)

| Output | Cited as |
|---|---|
| `outputs/tables/md_statistical_tests.csv` | "Within each pair, every MD feature differs significantly between states (Mann-Whitney U, BH q<0.05)." |
| `outputs/tables/audio_statistical_tests.csv` | "Within each pair, X of 28 audio descriptors reproduce the separation." |
| `outputs/tables/binary_classification_results.csv` | "Activation pair: pooled audio RF balanced acc = A.AA; ligand pair: B.BB (chance = 0.50)." |
| `outputs/tables/loio_binary_classification_results.csv` | "LOIO mean balanced acc — activation pair: C.CC; ligand pair: D.DD." |
| `outputs/tables/md_baseline_classification_results.csv` | "Per-pair MD-feature baselines (upper bound)." |
| `outputs/tables/information_retention.csv` | "Information-retention ratio (audio/MD): activation R₁; ligand R₂." |
| `outputs/tables/cca_md_audio.csv` | "First canonical correlation — activation: r₁=E.EE; ligand: r₁'=F.FF." |
| `outputs/figures/fig_classification_confusion_per_pair.*` | **Main classification figure** (per-pair binary CMs) |
| `outputs/figures/fig_classification_loio_per_pair.*` | **Main LOIO figure** (2 pairs × 3 instruments) |
| `outputs/figures/fig_cca_md_audio.*` | **Main CCA figure** (per-pair bars) |

### Supplementary outputs (3-class)

| Output | Note |
|---|---|
| `outputs/tables/three_class_classification_results.csv` | 3-class RF, chance = 0.333 |
| `outputs/tables/loio_3class_classification_results.csv` | 3-class LOIO |
| `outputs/tables/md_baseline_classification_results.csv` (row `3_class`) | 3-class MD baseline |
| `outputs/tables/cca_md_audio.csv` (row `3_state`) | 3-state CCA |
| `outputs/figures/fig_classification_confusion_3class_pooled.*` | 3-class pooled CM |
| `outputs/figures/fig_audio_features_pca.*` | 3-class PCA (visual orientation) |

### Manuscript claim that these results support

> *Under an identical sonification mapping rendered with three timbres,
> the auditory representation of β2AR molecular dynamics preserves the
> MD-state information sufficient to discriminate both designed
> contrasts — the **activation pair** (inactive vs active) and the
> **ligand pair** (active apo vs active + agonist) — at well above
> chance level (0.500), with timbre-independent generalisation under
> leave-one-instrument-out testing and a first canonical correlation
> approaching the MD-feature baseline. Sonification therefore acts as a
> quantitatively faithful complementary perceptual representation of
> GPCR conformational dynamics.*
